# Machine Learning 1 – Práctica Clase 9: Árboles de Decisión, Poda, Ensambles (Bagging & Boosting), Optuna y Reglas de Asociación
### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)
**Profesor:** Dr. Diego Stalder | **Auxiliar de Cátedra:** Ing. Carlos Benitez

---

### Objetivos Pedagógicos de la Clase:
1. **Árboles de Decisión (CART & ID3):** Comprender los criterios matemáticos de división (**Entropía** y **Ganancia de Información** vs. **Impureza de Gini**), visualización de árboles y análisis de *Feature Importance*.
2. **Árboles para Regresión:** Modelar problemas continuos mediante particionamiento recursivo del espacio y aproximación por funciones escalonadas.
3. **Control del Sobreajuste (Overfitting):** Implementar **Pre-pruning** (restricciones heurísticas) y **Post-pruning** formal mediante **Poda por Complejidad de Costo (*Cost-Complexity Pruning - CCP*)**.
4. **Métodos de Ensamble:** Comparar arquitecturas de **Bagging** (*Random Forest*) y **Boosting** (*AdaBoost*, *Gradient Boosting*, *XGBoost* y *LightGBM*) evaluando rendimiento, curvas ROC y velocidad de cómputo.
5. **Optimización Bayesiana con Optuna:** Definir funciones objetivo, espacios de búsqueda y optimizar hiperparámetros de modelos basados en árboles mediante TPE (*Tree-structured Parzen Estimator*).
6. **Algoritmo Apriori:** Comprender la minería de reglas de asociación con métricas de Soporte, Confianza y Lift.
7. **Guía para el TP5:** Estructurar los experimentos requeridos para el Trabajo Práctico 5 del curso.


---
## 0. Configuración del Entorno e Instalación de Dependencias

Aseguramos la disponibilidad de las librerías científicas, de optimización y ensambles acelerados por gradiente.


In [ ]:
# Instalación de librerías necesarias en entornos como Google Colab o Jupyter local
# Descomenta la siguiente línea si necesitas instalarlas:
# !pip install xgboost lightgbm optuna mlxtend shap


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn: Árboles y Regresión
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree,
    export_text
)

# Scikit-Learn: Métodos de Ensamble
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

# Scikit-Learn: Selección de Modelos y Métricas
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    mean_squared_error
)

# Datasets de prueba
from sklearn.datasets import load_breast_cancer, make_moons

# Ensambles Modernos
try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

# Optimización de Hiperparámetros
try:
    import optuna
    from optuna.visualization.matplotlib import (
        plot_optimization_history,
        plot_param_importances,
        plot_slice
    )
except ImportError:
    optuna = None

# Configuración visual global
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('tab10')
print("✅ Entorno e importaciones configuradas exitosamente.")


---
## 1. Fundamentos Matemáticos: Entropía vs. Impureza de Gini (ID3 vs. CART)

Un árbol de decisión busca particionar recursivamente el espacio de características de manera tal que los subconjuntos resultantes sean lo más **puros** posible respecto a la variable objetivo $y$.

### 1.1. Criterios de Impureza

#### A. Entropía de Shannon (Algoritmo ID3 / C4.5)
Mide la incertidumbre o desorden en un conjunto de datos $S$:
$$H(S) = - \sum_{i=1}^{C} p_i \log_2(p_i)$$
donde $p_i$ es la proporción de muestras que pertenecen a la clase $i$, y $C$ es el número total de clases. Si todas las muestras pertenecen a la misma clase, $H(S) = 0$ (pureza máxima). Si están uniformemente distribuidas entre dos clases ($p_1 = p_2 = 0.5$), $H(S) = 1$ bit (máxima impureza).

La **Ganancia de Información** (*Information Gain*) al realizar una partición basada en la característica $A$ se define como:
$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

#### B. Impureza de Gini (Algoritmo CART - Breiman et al.)
Mide la probabilidad de que una muestra elegida aleatoriamente sea clasificada erróneamente si se le asigna una etiqueta según la distribución de clases del subconjunto:
$$Gini(S) = 1 - \sum_{i=1}^{C} p_i^2 = \sum_{i=1}^{C} p_i (1 - p_i)$$
Para un problema binario equilibrado ($p_1 = p_2 = 0.5$), $Gini(S) = 1 - (0.25 + 0.25) = 0.5$.

> **Comparación Práctica:**
> - **Gini** es el estándar por defecto en Scikit-Learn porque no requiere calcular funciones logarítmicas, resultando **computacionalmente más veloz**.
> - En el 98% de los casos prácticos, ambos criterios producen árboles con estructuras y precisiones prácticamente indistinguibles.


In [ ]:
# Implementación manual de Entropía y Gini para entender su cálculo interno
def entropia_manual(y):
    clases, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def gini_manual(y):
    clases, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1.0 - np.sum(p ** 2)

# Demostración con el dataset de juguete de la teoría: 'vous' (1) vs 'tu' (0)
# Regla sociocultural: Uso de 'vous' o 'tu' según Edad, Amistad y Género
data_toy = {
    'Age':    [1, 1, 1, 1, 0],  # 1 = Adulto, 0 = Menor
    'Friend': [0, 0, 1, 1, 0],  # 1 = Amigo,  0 = Desconocido
    'Gender': [1, 0, 1, 0, 1],  # 1 = M,      0 = F
    'Target': [1, 1, 0, 0, 0]   # 1 = vous,   0 = tu
}
df_toy = pd.DataFrame(data_toy)
X_toy = df_toy[['Age', 'Friend', 'Gender']]
y_toy = df_toy['Target']

print("=== DATASET DE JUGUETE (vous vs tu) ===")
display(df_toy)
print(f"Entropía Inicial del Nodo Raíz: {entropia_manual(y_toy):.4f} bits")
print(f"Impureza de Gini Inicial:        {gini_manual(y_toy):.4f}")


### 1.2. Comparación Lado a Lado: Árbol con Entropía vs. Árbol con Gini

Entrenamos dos modelos `DecisionTreeClassifier` para verificar visualmente cómo dividen los datos y constatar si coinciden en la raíz.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Modelo con Entropía (Information Gain)
tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
tree_entropy.fit(X_toy, y_toy)
plot_tree(tree_entropy, feature_names=list(X_toy.columns), class_names=['tu (0)', 'vous (1)'],
          filled=True, rounded=True, ax=axes[0])
axes[0].set_title("Criterio: Entropía (Information Gain)", fontsize=13, fontweight='bold')

# Modelo con Gini (CART)
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree_gini.fit(X_toy, y_toy)
plot_tree(tree_gini, feature_names=list(X_toy.columns), class_names=['tu (0)', 'vous (1)'],
          filled=True, rounded=True, ax=axes[1])
axes[1].set_title("Criterio: Gini Impurity (CART Default)", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Representación en texto de las reglas aprendidas
print("Reglas del Árbol de Decisión (Texto Plano):")
print(export_text(tree_entropy, feature_names=list(X_toy.columns)))


---
## 2. Árboles de Regresión (CART Regressor)

El algoritmo CART (*Classification And Regression Trees*) no se limita a la clasificación categórica; también modela variables dependientes continuas.

### ¿Cómo funciona la regresión con árboles?
1. En lugar de impureza de Gini/Entropía, el criterio de división es la **minimización del Error Cuadrático Medio ($MSE$)** o de la varianza:
   $$MSE(S) = \frac{1}{|S|} \sum_{i \in S} (y_i - \bar{y}_S)^2$$
   donde $\bar{y}_S = \frac{1}{|S|} \sum_{i \in S} y_i$ es el valor medio de la variable objetivo en el nodo $S$.
2. La predicción para cualquier muestra nueva que caiga en una hoja terminal $R_m$ es simplemente el promedio de las muestras de entrenamiento en esa hoja: $\hat{y} = \bar{y}_m$.
3. Esto genera una función de aproximación **escalonada por tramos (*piecewise constant*)**.


In [ ]:
# Demostración de Árbol de Regresión con datos sintéticos no lineales ruidosos
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 10, size=(120, 1)), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, size=X_reg.shape[0])

# Ajustar árboles con diferentes niveles de profundidad
depths = [2, 5, None]
colors = ['forestgreen', 'darkorange', 'crimson']

plt.figure(figsize=(14, 6))
plt.scatter(X_reg, y_reg, s=35, edgecolor='black', c='lightgray', label='Datos con Ruido $y = \sin(x) + \epsilon$')

X_grid = np.linspace(0, 10, 500)[:, np.newaxis]

for d, c in zip(depths, colors):
    reg = DecisionTreeRegressor(max_depth=d, random_state=42)
    reg.fit(X_reg, y_reg)
    y_pred_grid = reg.predict(X_grid)
    mse = mean_squared_error(y_reg, reg.predict(X_reg))
    label_txt = f'max_depth={d} (MSE: {mse:.3f})' if d else f'max_depth=None (Overfitting, MSE: {mse:.3f})'
    plt.plot(X_grid, y_pred_grid, color=c, lw=2.2, label=label_txt)

plt.title("Árbol de Decisión para Regresión: Efecto de la Profundidad en la Curva Escalonada", fontsize=13, fontweight='bold')
plt.xlabel("Característica $X$")
plt.ylabel("Objetivo Continuo $y$")
plt.legend(frameon=True)
plt.show()


---
## 3. Importancia de Características (*Feature Importance*)

En Scikit-Learn, la importancia de una característica en un árbol de decisión se calcula mediante la **Reducción Media de la Impureza (*Mean Decrease in Impurity - MDI*)**.
Para cada característica, se suma la reducción ponderada de impureza que genera en todos los nodos donde fue utilizada para realizar la partición:

$$I(A) = \sum_{t \in T : v(t) = A} \frac{N_t}{N} \left[ i(t) - \frac{N_{t_L}}{N_t} i(t_L) - \frac{N_{t_R}}{N_t} i(t_R) \right]$$

Los valores se normalizan de forma tal que sumen $1.0$ (o 100%).


In [ ]:
# Extracción y visualización de la importancia de características
importances = tree_toy.feature_importances_
feature_names = X_toy.columns

df_imp = pd.DataFrame({'Característica': feature_names, 'Importancia': importances}).sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=df_imp, x='Importancia', y='Característica', palette='viridis')
plt.title("Feature Importance (MDI) en el Árbol de vous vs tu", fontsize=12, fontweight='bold')
plt.xlabel("Reducción Normalizada de la Impureza")
plt.xlim(0, 1.0)
for idx, row in df_imp.iterrows():
    plt.text(row['Importancia'] + 0.02, idx, f"{row['Importancia']:.2%}", va='center', fontweight='bold')
plt.show()

print("Interpretación: La variable 'Friend' resuelve por sí sola la mayor cantidad de incertidumbre.")


---
## 4. Control del Sobreajuste: Pre-pruning vs. Post-pruning (Cost-Complexity Pruning)

Los árboles de decisión no restringidos tienden a memorizar el ruido de los datos, produciendo fronteras de decisión excesivamente fragmentadas y sobreajustadas.

Existen dos estrategias principales para controlar este fenómeno:
1. **Pre-pruning (Poda Preventiva o Temprana):** Detiene el crecimiento del árbol antes de que alcance su desarrollo total, mediante hiperparámetros como `max_depth`, `min_samples_split`, o `min_samples_leaf`.
2. **Post-pruning (Poda Posterior / Cost-Complexity Pruning - CCP):** Deja crecer el árbol completamente y luego poda recursivamente las ramas que aportan poco poder predictivo respecto a la complejidad que añaden.

### 4.1. Visualización con Dataset Sintético Ruidoso (`make_moons`)


In [ ]:
# Generamos un dataset bidimensional con frontera no lineal y ruido
X_moons, y_moons = make_moons(n_samples=400, noise=0.35, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42, stratify=y_moons
)

def plot_decision_boundary(clf, X, y, title, ax=None):
    if ax is None:
        ax = plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='coolwarm', s=35)
    ax.set_title(title, fontsize=11, fontweight='bold')
    return scatter

# Comparación: Árbol no restringido (Overfit) vs Árbol con Pre-pruning
tree_unpruned = DecisionTreeClassifier(random_state=42)
tree_unpruned.fit(X_train_m, y_train_m)

tree_prepruned = DecisionTreeClassifier(max_depth=4, min_samples_leaf=6, random_state=42)
tree_prepruned.fit(X_train_m, y_train_m)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(tree_unpruned, X_test_m, y_test_m,
                       f"Sin Poda (Profundidad Libre)\nTrain: {tree_unpruned.score(X_train_m, y_train_m):.2f} | Test: {tree_unpruned.score(X_test_m, y_test_m):.2f}",
                       ax=axes[0])
plot_decision_boundary(tree_prepruned, X_test_m, y_test_m,
                       f"Con Pre-poda (max_depth=4, min_samples_leaf=6)\nTrain: {tree_prepruned.score(X_train_m, y_train_m):.2f} | Test: {tree_prepruned.score(X_test_m, y_test_m):.2f}",
                       ax=axes[1])
plt.tight_layout()
plt.show()


### 4.2. Post-pruning Formal: Cost-Complexity Pruning (CCP con `ccp_alpha`)

El algoritmo CART introduce el parámetro de complejidad de costo $\alpha \ge 0$ para penalizar el tamaño del árbol en la función de costo:
$$R_\alpha(T) = R(T) + \alpha |T|$$
donde:
- $R(T)$ es la tasa total de error (impureza) del árbol $T$.
- $|T|$ es el número de hojas terminales.
- $\alpha$ controla el equilibrio entre precisión y tamaño: cuando $\alpha = 0$, se obtiene el árbol completo sin podar; a medida que $\alpha$ aumenta, más hojas son podadas hasta reducir el árbol a un único nodo raíz.

Scikit-Learn permite obtener toda la trayectoria de alphas efectivos mediante `cost_complexity_pruning_path`.


In [ ]:
# 1. Obtener la trayectoria de alphas efectivos de poda
path = tree_unpruned.cost_complexity_pruning_path(X_train_m, y_train_m)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

print(f"Número de valores de ccp_alpha evaluados en la ruta: {len(ccp_alphas)}")

# 2. Entrenar un árbol para cada alpha de la trayectoria
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    clf.fit(X_train_m, y_train_m)
    clfs.append(clf)

# Removemos el último elemento (que corresponde al árbol trivial de 1 solo nodo)
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

# 3. Evaluar el número de nodos, profundidad y Accuracy en Train vs Test
node_counts = [clf.tree_.node_count for clf in clfs]
depths = [clf.tree_.max_depth for clf in clfs]
train_scores = [clf.score(X_train_m, y_train_m) for clf in clfs]
test_scores = [clf.score(X_test_m, y_test_m) for clf in clfs]

# 4. Graficar la evolución
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Número de nodos vs alpha
axes[0].plot(ccp_alphas, node_counts, marker='o', drawstyle="steps-post", color='purple')
axes[0].set_xlabel("Parámetro de Poda $\\alpha$ (ccp_alpha)")
axes[0].set_ylabel("Número Total de Nodos")
axes[0].set_title("Reducción de Complejidad del Árbol en función de $\\alpha$", fontweight='bold')

# Gráfico 2: Accuracy vs alpha
axes[1].plot(ccp_alphas, train_scores, marker='o', label="Train Accuracy", drawstyle="steps-post", color='royalblue')
axes[1].plot(ccp_alphas, test_scores, marker='s', label="Test Accuracy", drawstyle="steps-post", color='crimson')
axes[1].set_xlabel("Parámetro de Poda $\\alpha$ (ccp_alpha)")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs $\\alpha$ (Búsqueda del $\\alpha$ Óptimo)", fontweight='bold')
axes[1].legend()

# Identificar el mejor alpha para el conjunto de test
best_idx = np.argmax(test_scores)
best_alpha = ccp_alphas[best_idx]
axes[1].axvline(best_alpha, color='black', linestyle='--', alpha=0.7, label=f'Óptimo $\\alpha={best_alpha:.4f}$')
axes[1].legend()

plt.tight_layout()
plt.show()

# 5. Entrenar el árbol podado óptimo y comparar
best_tree_ccp = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
best_tree_ccp.fit(X_train_m, y_train_m)

plt.figure(figsize=(7, 5))
plot_decision_boundary(best_tree_ccp, X_test_m, y_test_m,
                       f"Árbol con Post-Poda Óptima (ccp_alpha={best_alpha:.4f})\nTest Accuracy: {best_tree_ccp.score(X_test_m, y_test_m):.2%}")
plt.show()


---
## 5. Métodos de Ensamble: Bagging vs. Boosting

Los árboles individuales sufren de **alta varianza** (pequeños cambios en los datos producen árboles muy distintos). Para construir modelos robustos y de clase mundial, combinamos múltiples árboles mediante **Ensambles**:

### 5.1. Paradigmas de Ensamble
| Característica | Bagging (Bootstrap Aggregating) | Boosting (Aceleración de Gradiente) |
| :--- | :--- | :--- |
| **Algoritmo Representativo** | **Random Forest** | **AdaBoost, Gradient Boosting, XGBoost, LightGBM** |
| **Entrenamiento** | **Paralelo:** Árboles independientes construidos con réplicas bootstrap. | **Secuencial:** Cada árbol se construye para corregir los errores o residuos del anterior. |
| **Mecanismo Clave** | Promedio de árboles descorrelacionados (*Random Subspaces*). | Ajuste paso a paso mediante descenso de gradiente en el espacio de funciones. |
| **Objetivo Principal** | **Reducir la Varianza** manteniendo bajo el sesgo. | **Reducir el Sesgo** y mejorar la precisión predictiva. |

### 5.2. Comparación Práctica en el Dataset de Cáncer de Mama (`load_breast_cancer`)
Evaluaremos los 5 algoritmos del plan:
1. **Random Forest** (Bagging)
2. **AdaBoost** (Boosting adaptativo)
3. **Gradient Boosting** (Boosting clásico de Scikit-Learn)
4. **XGBoost** (Extreme Gradient Boosting optimizado)
5. **LightGBM** (Light Gradient Boosting Machine, rápido y eficiente en memoria)


In [ ]:
# Cargar dataset clínico real
data_bc = load_breast_cancer()
X_bc = data_bc.data
y_bc = data_bc.target
feature_names_bc = data_bc.feature_names
target_names_bc = data_bc.target_names

# División estratificada 75% entrenamiento / 25% prueba
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc
)

print(f"Dimensiones del dataset: {X_bc.shape[0]} muestras, {X_bc.shape[1]} características.")
print(f"Distribución de clases: {np.bincount(y_bc)} (0: Maligno, 1: Benigno)")


In [ ]:
# Configuración de los 5 modelos de ensamble
candidate_models = {
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost (Boosting)': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting (Sklearn)': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

if xgb is not None:
    candidate_models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100, eval_metric='logloss', random_state=42, use_label_encoder=False
    )

if lgb is not None:
    candidate_models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=100, random_state=42, verbose=-1
    )

# Evaluación sistemática de métricas y tiempos
benchmark_results = []
trained_models = {}

for name, clf in candidate_models.items():
    # Medición de tiempo de entrenamiento
    t0 = time()
    clf.fit(X_tr_bc, y_tr_bc)
    fit_time = time() - t0
    
    # Medición de tiempo de inferencia
    t1 = time()
    y_pred = clf.predict(X_te_bc)
    pred_time = time() - t1
    
    # Probabilidades para ROC-AUC
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_te_bc)[:, 1]
    else:
        y_proba = clf.decision_function(X_te_bc)
        
    acc = accuracy_score(y_te_bc, y_pred)
    f1 = f1_score(y_te_bc, y_pred, average='weighted')
    auc = roc_auc_score(y_te_bc, y_proba)
    
    benchmark_results.append({
        'Modelo': name,
        'Accuracy Test': acc,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'Tiempo Fit (s)': fit_time,
        'Tiempo Pred (ms)': pred_time * 1000
    })
    trained_models[name] = clf

df_benchmark = pd.DataFrame(benchmark_results).sort_values(by='ROC-AUC', ascending=False)
display(df_benchmark.style.highlight_max(subset=['Accuracy Test', 'F1-Score', 'ROC-AUC'], color='lightgreen')
                         .highlight_min(subset=['Tiempo Fit (s)'], color='lightblue'))


In [ ]:
# Visualización comparativa de Rendimiento vs Tiempo de Entrenamiento
fig, ax1 = plt.subplots(figsize=(10, 5))

# Barras para ROC-AUC
x_pos = np.arange(len(df_benchmark))
ax1.bar(x_pos - 0.15, df_benchmark['ROC-AUC'], width=0.3, label='ROC-AUC', color='teal', alpha=0.85)
ax1.bar(x_pos + 0.15, df_benchmark['Accuracy Test'], width=0.3, label='Accuracy', color='cornflowerblue', alpha=0.85)
ax1.set_ylabel("Métricas de Rendimiento", fontsize=11, fontweight='bold')
ax1.set_ylim(0.90, 1.0)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(df_benchmark['Modelo'], rotation=15, ha='right')

# Eje secundario para tiempo de entrenamiento
ax2 = ax1.twinx()
ax2.plot(x_pos, df_benchmark['Tiempo Fit (s)'], color='crimson', marker='o', lw=2.5, label='Tiempo Entrenamiento (s)')
ax2.set_ylabel("Tiempo de Ajuste (segundos)", color='crimson', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='crimson')

plt.title("Comparación Multidimensional de Ensambles en Cáncer de Mama", fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()


---
## 6. Evaluación Diagnóstica: Matrices de Confusión y Curvas ROC

En problemas clínicos o de ingeniería de alta criticidad, la exactitud global (*Accuracy*) no es suficiente. Debemos inspeccionar la **Matriz de Confusión** (evitando falsos negativos) y las **Curvas ROC** con sus respectivas áreas bajo la curva (AUC).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Curvas ROC para todos los modelos
for name, clf in candidate_models.items():
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X_te_bc)[:, 1]
    else:
        y_proba = clf.decision_function(X_te_bc)
    fpr, tpr, _ = roc_curve(y_te_bc, y_proba)
    auc_val = roc_auc_score(y_te_bc, y_proba)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[0].set_ylabel("Tasa de Verdaderos Positivos (TPR - Recall)")
axes[0].set_title("Curvas ROC Comparativas", fontweight='bold')
axes[0].legend(loc='lower right')

# 2. Matriz de Confusión del Mejor Modelo
best_model_name = df_benchmark.iloc[0]['Modelo']
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_te_bc)

cm = confusion_matrix(y_te_bc, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Maligno', 'Benigno'])
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f"Matriz de Confusión: {best_model_name}", fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.show()

print(f"=== Reporte Detallado de Clasificación ({best_model_name}) ===")
print(classification_report(y_te_bc, y_pred_best, target_names=['Maligno', 'Benigno']))


---
## 7. Interpretabilidad y Explicabilidad: Feature Importance y SHAP

El **Plan de Clases** estipula el análisis de interpretabilidad mediante *Feature Importance* y **SHAP (SHapley Additive exPlanations)** para los modelos avanzados.

### 7.1. Comparación de Importancia de Características entre Modelos
Diferentes algoritmos pueden priorizar variables distintas debido a su dinámica de división (Bagging promedia subespacios aleatorios, mientras que Boosting focaliza residuos).


In [ ]:
# Comparación de las 10 características más relevantes entre Random Forest y LightGBM / Gradient Boosting
top_n = 10
rf_clf = candidate_models['Random Forest (Bagging)']
boost_clf = candidate_models.get('LightGBM', candidate_models.get('XGBoost', candidate_models['Gradient Boosting (Sklearn)']))

importances_rf = rf_clf.feature_importances_
importances_boost = boost_clf.feature_importances_ / boost_clf.feature_importances_.sum()

top_idx_rf = np.argsort(importances_rf)[-top_n:]

df_comp_imp = pd.DataFrame({
    'Característica': feature_names_bc[top_idx_rf],
    'Random Forest': importances_rf[top_idx_rf],
    'Boosting': importances_boost[top_idx_rf]
}).melt(id_vars='Característica', var_name='Algoritmo', value_name='Importancia')

plt.figure(figsize=(10, 6))
sns.barplot(data=df_comp_imp, y='Característica', x='Importancia', hue='Algoritmo', palette='Set2')
plt.title(f"Top {top_n} Características más Influyentes: Bagging vs Boosting", fontweight='bold')
plt.xlabel("Importancia Normalizada")
plt.show()


### 7.2. Introducción Teórica y Práctica a SHAP (*SHapley Additive exPlanations*)

Basado en la **Teoría de Juegos Cooperativos de Lloyd Shapley (Premio Nobel)**, los valores SHAP representan la contribución marginal justa de cada característica a la predicción particular de una muestra:

$$f(x) = \phi_0 + \sum_{j=1}^{M} \phi_j(x)$$
donde $\phi_0$ es el valor base esperado del modelo (promedio sobre todo el dataset) y $\phi_j(x)$ es el valor Shapley de la variable $j$.

- **A diferencia del MDI**, SHAP es **consistente** (no sobreestima variables de alta cardinalidad o continuas con ruido) e indica no solo la magnitud sino la **dirección del impacto** (si aumenta o disminuye la probabilidad de la clase).


In [ ]:
# Demostración conceptual de SHAP (con TreeExplainer si la librería está instalada)
try:
    import shap
    print("Calculando valores SHAP con TreeExplainer...")
    explainer = shap.TreeExplainer(rf_clf)
    shap_values = explainer.shap_values(X_te_bc)
    
    # Para clasificación binaria, tomamos los valores SHAP correspondientes a la clase 1
    vals = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(vals, X_te_bc, feature_names=feature_names_bc, max_display=10, show=False)
    plt.title("SHAP Summary Plot: Impacto de cada característica en la predicción", fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("ℹ️ La librería 'shap' no está instalada en este entorno.")
    print("Para utilizarla en Google Colab o tu entorno local, ejecuta: !pip install shap")


---
## 8. Optimización de Hiperparámetros con Optuna

**Optuna** es el framework moderno estándar para optimización de hiperparámetros. Utiliza algoritmos de optimización bayesiana basados en **TPE (*Tree-structured Parzen Estimator*)**, que modelan probabilísticamente las regiones del espacio de búsqueda con mayor probabilidad de maximizar la métrica objetivo.

A diferencia de `GridSearchCV` (fuerza bruta exhaustiva) o `RandomizedSearchCV` (muestreo ciego), Optuna:
1. Aprende activamente de las pruebas anteriores.
2. Permite poda temprana de ensayos no prometedores (*Pruning*).
3. Es agnóstico al framework y extraordinariamente flexible.


In [ ]:
# Definición de la función objetivo con validación cruzada estratificada de 5 pliegues
def objetivo_optuna(trial):
    # Definición del espacio de búsqueda para LightGBM (o Gradient Boosting si LGBM no está presente)
    if lgb is not None:
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'verbosity': -1,
            'boosting_type': 'gbdt',
            'random_state': 42,
            'n_estimators': trial.suggest_int('n_estimators', 50, 200, step=25),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.25, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 25),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True)
        }
        modelo = lgb.LGBMClassifier(**params)
    else:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 150),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'random_state': 42
        }
        modelo = GradientBoostingClassifier(**params)
        
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score_cv = cross_val_score(modelo, X_tr_bc, y_tr_bc, cv=cv, scoring='roc_auc').mean()
    return score_cv

if optuna is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    
    print("⏳ Iniciando optimización bayesiana con Optuna (25 trials)...")
    t0_opt = time()
    study.optimize(objetivo_optuna, n_trials=25)
    print(f"✅ Optimización completada en {time() - t0_opt:.2f} segundos.")
    
    print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===")
    for k, v in study.best_params.items():
        print(f"  • {k}: {v}")
    print(f"\nMejor ROC-AUC en Validación Cruzada: {study.best_value:.4f}")
else:
    print("ℹ️ Optuna no está disponible en este entorno.")


### 8.1. Visualizaciones Analíticas del Estudio de Optuna


In [ ]:
if optuna is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 1. Historial de optimización
    plot_optimization_history(study, ax=axes[0])
    axes[0].set_title("Historial de Convergencia del Estudio", fontweight='bold')
    
    # 2. Importancia de los hiperparámetros
    plot_param_importances(study, ax=axes[1])
    axes[1].set_title("Importancia Relativa de los Hiperparámetros", fontweight='bold')
    
    plt.tight_layout()
    plt.show()


### 8.2. Evaluación del Modelo Final Optimizado vs. Modelo Baseline en Test


In [ ]:
if optuna is not None:
    # 1. Entrenar el modelo final con los mejores parámetros descubiertos
    if lgb is not None:
        best_opt_model = lgb.LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
        base_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
    else:
        best_opt_model = GradientBoostingClassifier(**study.best_params, random_state=42)
        base_model = GradientBoostingClassifier(random_state=42)
        
    base_model.fit(X_tr_bc, y_tr_bc)
    best_opt_model.fit(X_tr_bc, y_tr_bc)
    
    auc_base = roc_auc_score(y_te_bc, base_model.predict_proba(X_te_bc)[:, 1])
    auc_opt = roc_auc_score(y_te_bc, best_opt_model.predict_proba(X_te_bc)[:, 1])
    
    acc_base = accuracy_score(y_te_bc, base_model.predict(X_te_bc))
    acc_opt = accuracy_score(y_te_bc, best_opt_model.predict(X_te_bc))
    
    df_opt_eval = pd.DataFrame([
        {'Versión': 'Baseline (Hiperparámetros por Defecto)', 'Accuracy Test': acc_base, 'ROC-AUC Test': auc_base},
        {'Versión': 'Optimizado con Optuna (TPE)', 'Accuracy Test': acc_opt, 'ROC-AUC Test': auc_opt}
    ])
    
    print("=== IMPACTO DE LA OPTIMIZACIÓN EN EL CONJUNTO DE PRUEBA INDEPENDIENTE ===")
    display(df_opt_eval)


---
## 9. Reglas de Asociación: Algoritmo Apriori (Semana 13 del Plan)

El **Plan de Clases** contempla en la Semana 13 el estudio del **Algoritmo Apriori**, una técnica clásica de aprendizaje no supervisado aplicada en *Market Basket Analysis* (Análisis de la Cesta de la Compra).

### Métricas Fundamentales de Asociación:
Para una regla de la forma $A \Rightarrow B$ (donde $A$ es el antecedente y $B$ el consecuente):

1. **Soporte (Support):** Frecuencia relativa con la que los ítems $A$ y $B$ aparecen juntos en las transacciones:
   $$\text{Support}(A \Rightarrow B) = P(A \cup B) = \frac{\text{Transacciones conteniendo } A \text{ y } B}{\text{Total de Transacciones}}$$
2. **Confianza (Confidence):** Certeza condicional de que ocurra $B$ dado que ocurrió $A$:
   $$\text{Confidence}(A \Rightarrow B) = P(B|A) = \frac{P(A \cup B)}{P(A)} = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$
3. **Elevación (Lift):** Mide la fuerza de la regla respecto a la independencia estadística:
   $$\text{Lift}(A \Rightarrow B) = \frac{P(A \cup B)}{P(A) \cdot P(B)} = \frac{\text{Confidence}(A \Rightarrow B)}{\text{Support}(B)}$$
   - $\text{Lift} = 1$: $A$ y $B$ son estadísticamente independientes.
   - $\text{Lift} > 1$: Asociación **positiva** fuerte (la compra de $A$ incrementa la probabilidad de compra de $B$).
   - $\text{Lift} < 1$: Asociación **negativa** (la compra de $A$ desincentiva la de $B$).


In [ ]:
# Demostración del Algoritmo Apriori
# Dataset sintético de transacciones de un supermercado
dataset_transacciones = [
    ['Leche', 'Pan', 'Mantequilla'],
    ['Cerveza', 'Pan'],
    ['Leche', 'Pan', 'Mantequilla', 'Cerveza'],
    ['Leche', 'Cafe'],
    ['Cafe', 'Pan'],
    ['Leche', 'Pan', 'Mantequilla', 'Cafe'],
    ['Cerveza', 'Papas'],
    ['Pan', 'Mantequilla'],
    ['Leche', 'Pan'],
    ['Leche', 'Cafe', 'Pan', 'Mantequilla']
]

try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
    
    te = TransactionEncoder()
    te_ary = te.fit(dataset_transacciones).transform(dataset_transacciones)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    print("Matriz Binaria de Transacciones:")
    display(df_trans.head())
    
    # 1. Obtener itemsets frecuentes con soporte mínimo de 30%
    frequent_itemsets = apriori(df_trans, min_support=0.3, use_colnames=True)
    frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
    
    print("\nItemsets Frecuentes Encontrados:")
    display(frequent_itemsets.sort_values(by='support', ascending=False))
    
    # 2. Generar reglas de asociación con confianza mínima de 60%
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
    rules_filtered = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)
    
    print("\nReglas de Asociación con Mayor Lift:")
    display(rules_filtered)
    
except ImportError:
    print("ℹ️ La librería 'mlxtend' no está instalada.")
    print("Para ejecutar minería de reglas de asociación con Apriori: !pip install mlxtend")


---
## 10. Guía de Conexión con el Trabajo Práctico 5 (TP5)

De acuerdo con el **Cuadro 2 del Plan de Clases**, el **TP5** consiste en:
> **"TP5: Optuna con comparación de modelos de decisión"**

### Requisitos y Pautas Sugeridas para el TP5:
1. **Selección de Dataset:** Escoger un dataset de ingeniería o ciencia de datos de complejidad real (disponible en Kaggle o UCI Machine Learning Repository).
2. **Modelos Requeridos:**
   - Un árbol de decisión podado (`DecisionTreeClassifier` o `DecisionTreeRegressor` con análisis de `ccp_alpha`).
   - Un modelo de Bagging (`RandomForestClassifier` o `RandomForestRegressor`).
   - Dos modelos de Boosting (`XGBoost` y `LightGBM` o `GradientBoostingClassifier`).
3. **Optimización con Optuna:**
   - Seleccionar el modelo con mejor desempeño preliminar.
   - Definir un espacio de búsqueda que incluya al menos 5 hiperparámetros críticos (`learning_rate`, `n_estimators`, `max_depth`, regularizadores L1/L2, `subsample`).
   - Ejecutar un estudio de al menos 30 ensayos con validación cruzada.
4. **Análisis Crítico:**
   - Construir una tabla comparativa antes y después de la optimización.
   - Generar gráficos de `plot_optimization_history` y `plot_param_importances`.
   - Analizar las matrices de confusión y curvas ROC.
   - Conclusiones justificadas desde la perspectiva de costo computacional vs. ganancia en exactitud.
